# 04 – Outliers, scaling, validation & metrics (M4) – Regression
**Target:** `Days for shipping (real)`. Uses M1's `train_reg.csv`, M2's `src/features.py`, M3's `src/encoders.py` and `results/m3_selection.json`, and M4's `src/numeric_prep.py` + `src/evaluate.py`.
Outputs: `results/m4_preprocessing.json`, rows in `results/experiments.csv`, charts in `reports/figures/m4/`.

## Step 0 – Setup

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "reports" / "figures" / "m4"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Days for shipping (real)"
GROUP = "Order Id"
SEED = 42
KNN_ORDER_FRAC = 0.3      # KNN is slow on 100k+ rows, so KNN experiments use 30% of the orders

def show(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120)
    plt.show()

## Step 1 – Load the data, M3's decision and the folds

In [ ]:
from src.features import DATE_COL
from src.encoders import ENGINEERED, split_columns
from src.evaluate import METRICS, make_cv, cv_evaluate, regression_metrics, log_experiment
from src.numeric_prep import build_full_pipeline, QuantileClipper, SkewedLog

train = pd.read_csv(PROCESSED_DIR / "train_reg.csv")
X, y, groups = train.drop(columns=[TARGET]), train[TARGET], train[GROUP]

sel_path = RESULTS_DIR / "m3_selection.json"
if sel_path.exists():
    m3 = json.loads(sel_path.read_text(encoding="utf-8"))
    HIGH_CARD = m3["high_card_strategy"]
    SELECTED = m3["selected_features"]
    FEATURES = SELECTED          # treatments are compared on the features the model will actually use
else:
    HIGH_CARD = "target"
    FEATURES = [c for c in X.columns if c not in (GROUP, DATE_COL)] + [u for u in ENGINEERED if all(c in X.columns for c in ENGINEERED[u])]
    SELECTED = FEATURES

NUMERIC, _, _ = split_columns(X, FEATURES)
cv_splits = make_cv(X, y, groups, n_splits=5, seed=SEED)

print(f"Train: {len(X):,} rows | high-card strategy: {HIGH_CARD}")
print(f"M3 selected ({len(SELECTED)}): {SELECTED}")
print(f"Features used for M4 comparisons ({len(FEATURES)}), numeric: {NUMERIC}")

## Step 2 – Validation design

In [ ]:
fold_rows = []
for i, (tr, va) in enumerate(cv_splits):
    shared = set(groups.iloc[tr]) & set(groups.iloc[va])
    fold_rows.append({
        "fold": i, "train_rows": len(tr), "val_rows": len(va), "val_orders": groups.iloc[va].nunique(),
        "val_mean_days": y.iloc[va].mean(), "val_std_days": y.iloc[va].std(),
        "orders_in_both": len(shared),
    })
folds = pd.DataFrame(fold_rows)
print(f"Whole training set: mean {y.mean():.3f}, std {y.std():.3f}")
folds.round(3)

In [ ]:
dist = pd.DataFrame({f"fold {i}": y.iloc[va].value_counts(normalize=True).sort_index() * 100
                     for i, (_, va) in enumerate(cv_splits)})
dist["all train"] = y.value_counts(normalize=True).sort_index() * 100
ax = dist.plot(marker="o", figsize=(6, 3))
ax.set_xlabel("Real shipping days"); ax.set_ylabel("% of rows"); ax.set_title("Target distribution in each validation fold")
show(ax.figure, "01_fold_distributions")

In [ ]:
# Why grouping matters: the same KNN model scored with a plain (random) KFold vs the grouped folds
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor

knn_orders = pd.Series(groups.unique()).sample(frac=KNN_ORDER_FRAC, random_state=SEED)
k_mask = groups.isin(knn_orders).to_numpy()
Xk, yk, gk = X[k_mask].reset_index(drop=True), y[k_mask].reset_index(drop=True), groups[k_mask].reset_index(drop=True)
cv_k_grouped = make_cv(Xk, yk, gk, n_splits=5, seed=SEED)
cv_k_plain = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(Xk))

knn = build_full_pipeline(KNeighborsRegressor(n_neighbors=15, n_jobs=-1), Xk, FEATURES,
                          high_card=HIGH_CARD, outlier="none", scaler="standard")
plain_s, _ = cv_evaluate(knn, Xk, yk, cv_k_plain)
group_s, _ = cv_evaluate(knn, Xk, yk, cv_k_grouped)
pd.DataFrame({"plain KFold (leaky)": plain_s, "grouped (correct)": group_s}).loc[["MAE", "RMSE", "R2", "within_1_day_%"]].round(4)

## Step 3 – Outliers and skew

In [ ]:
def iqr_share(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean() * 100

skew_table = pd.DataFrame({
    "skew": [X[c].skew() for c in NUMERIC],
    "outliers_%": [iqr_share(X[c]) for c in NUMERIC],
    "negative_%": [(X[c] < 0).mean() * 100 for c in NUMERIC],
    "p1": [X[c].quantile(0.01) for c in NUMERIC],
    "p99": [X[c].quantile(0.99) for c in NUMERIC],
    "max": [X[c].max() for c in NUMERIC],
}, index=NUMERIC).sort_values("skew", key=abs, ascending=False)
skew_table["log_applied"] = skew_table["skew"].abs() > 1.0
skew_table.round(2)

In [ ]:
col = skew_table.index[0]                     # the most skewed numeric feature
logged = np.sign(X[col]) * np.log1p(X[col].abs())
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(X[col], bins=50); axes[0].set_title(f"{col} – raw (skew {X[col].skew():.2f})", fontsize=9)
axes[1].hist(logged, bins=50); axes[1].set_title(f"after signed log (skew {logged.skew():.2f})")
show(fig, "02_log_before_after")

In [ ]:
col = skew_table["outliers_%"].idxmax()        # the numeric feature with the most outliers
clipper = QuantileClipper(0.01, 0.99).fit(X[[col]])
clipped = pd.Series(clipper.transform(X[[col]]).ravel())
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(X[col], bins=50); axes[0].set_title(f"{col} – raw")
axes[1].hist(clipped, bins=50); axes[1].set_title(f"clipped to [{clipper.low_[0]:.0f}, {clipper.high_[0]:.0f}]")
show(fig, "03_clip_before_after")
print(f"Rows changed by clipping: {(clipped != X[col].to_numpy()).mean() * 100:.1f}%")

## Step 4 – Compare outlier treatment and scaling

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

OPTIONS = [("none", "none"), ("none", "standard"), ("none", "robust"), ("clip", "standard"), ("log", "standard")]
rows = []
for outlier, scaler in OPTIONS:
    s, _ = cv_evaluate(build_full_pipeline(Ridge(alpha=1.0), X, FEATURES, HIGH_CARD, outlier, scaler), X, y, cv_splits)
    rows.append({"model": "Ridge", "outlier": outlier, "scaler": scaler, **s})
    s, _ = cv_evaluate(build_full_pipeline(KNeighborsRegressor(n_neighbors=15, n_jobs=-1), Xk, FEATURES, HIGH_CARD, outlier, scaler),
                       Xk, yk, cv_k_grouped)
    rows.append({"model": "KNN (30% orders)", "outlier": outlier, "scaler": scaler, **s})

# Trees split on thresholds, so scaling should make no difference: check it once
for scaler in ["none", "standard"]:
    s, _ = cv_evaluate(build_full_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=SEED), X, FEATURES,
                                           HIGH_CARD, "none", scaler), X, y, cv_splits)
    rows.append({"model": "HistGB (tree)", "outlier": "none", "scaler": scaler, **s})

prep_results = pd.DataFrame(rows)
prep_results[["model", "outlier", "scaler", "MAE", "MAE_std", "RMSE", "R2", "within_1_day_%"]].round(4)

In [ ]:
prep_results["option"] = prep_results["outlier"] + " + " + prep_results["scaler"]
pivot = prep_results[prep_results["model"] != "HistGB (tree)"].pivot(index="option", columns="model", values="MAE")
pivot = pivot.loc[[f"{o} + {s}" for o, s in OPTIONS]]
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, m in zip(axes, pivot.columns):
    ax.bar(pivot.index, pivot[m]); ax.set_title(m); ax.set_ylabel("CV MAE (days)")
    ax.set_ylim(pivot[m].min() * 0.97, pivot[m].max() * 1.02); ax.tick_params(axis="x", rotation=45)
show(fig, "04_scaling_comparison")

In [ ]:
def best_option(model):
    r = prep_results[prep_results["model"] == model].sort_values("MAE").iloc[0]
    return {"outlier": r["outlier"], "scaler": r["scaler"], "cv_MAE": round(float(r["MAE"]), 4)}

CHOICE = {
    "linear": best_option("Ridge"),
    "distance (KNN)": best_option("KNN (30% orders)"),
    "tree": {"outlier": "none", "scaler": "none", "reason": "splits on thresholds, unaffected by scale or monotone transforms"},
}
tree = prep_results[prep_results["model"] == "HistGB (tree)"].set_index("scaler")["MAE"]
print(f"HistGB MAE without scaling {tree['none']:.4f} vs with scaling {tree['standard']:.4f}")
pd.DataFrame(CHOICE).T

## Step 5 – The metrics module in use

In [ ]:
from sklearn.dummy import DummyRegressor

baselines = {
    "Dummy (mean)": build_full_pipeline(DummyRegressor(strategy="mean"), X, SELECTED, HIGH_CARD),
    "Shipping Mode only": build_full_pipeline(Ridge(alpha=1.0), X, ["Shipping Mode"], HIGH_CARD, scaler="standard"),
}
base_rows, per_fold_all, oof = [], {}, {}
for name, pipe in baselines.items():
    summary, per_fold, oof[name] = cv_evaluate(pipe, X, y, cv_splits, return_predictions=True)
    per_fold_all[name] = per_fold
    base_rows.append({"baseline": name, **summary})
baseline_table = pd.DataFrame(base_rows).set_index("baseline")
baseline_table[METRICS + ["MAE_std"]].round(4)

In [ ]:
per_fold_all["Shipping Mode only"].round(4)

In [ ]:
err = oof["Shipping Mode only"] - y.to_numpy()
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(err, bins=np.arange(-6.5, 7, 0.25))
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Prediction error (days, + = predicted too slow)"); ax.set_ylabel("Rows")
ax.set_title("Out-of-fold errors of the Shipping Mode baseline")
show(fig, "05_baseline_errors")

## Step 6 – Save the decision and log the experiments

In [ ]:
decision = {
    "target": TARGET,
    "validation": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42), grouped by Order Id, stratified on the target",
    "metrics": METRICS,
    "primary_metric": "MAE",
    "preprocessing_by_model_family": CHOICE,
    "outlier_options": {"clip": "1st-99th percentile, learned on training folds",
                        "log": "signed log1p on columns with |skew| > 1 in training folds"},
    "baselines_cv_MAE": baseline_table["MAE"].round(4).to_dict(),
}
(RESULTS_DIR / "m4_preprocessing.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")

for _, r in prep_results.iterrows():
    log_experiment({"owner": "M4", "experiment": f"prep_{r['outlier']}_{r['scaler']}", "model": r["model"],
                    "n_features": len(FEATURES), **{m: r[m] for m in METRICS}, "MAE_std": r["MAE_std"]},
                   RESULTS_DIR / "experiments.csv")
for name, r in baseline_table.iterrows():
    log_experiment({"owner": "M4", "experiment": "baseline", "model": name, "n_features": len(SELECTED),
                    **{m: r[m] for m in METRICS}, "MAE_std": r["MAE_std"]}, RESULTS_DIR / "experiments.csv")
print("Saved m4_preprocessing.json; experiments.csv now has", len(pd.read_csv(RESULTS_DIR / "experiments.csv")), "rows")

## Step 7 – Final check on the test set (transform only)

In [ ]:
test = pd.read_csv(PROCESSED_DIR / "test_reg.csv")
X_te = test.drop(columns=[TARGET])
checks = {}
for family, opt in [("linear", CHOICE["linear"]), ("distance (KNN)", CHOICE["distance (KNN)"]), ("tree", CHOICE["tree"])]:
    pipe = build_full_pipeline(DummyRegressor(), X, SELECTED, HIGH_CARD, opt["outlier"], opt["scaler"])
    prep = pipe[:-1].fit(X, y)                           # everything except the model, fitted on TRAIN
    Xt_te = prep.transform(X_te)                         # test is only transformed
    checks[family] = {"outlier": opt["outlier"], "scaler": opt["scaler"], "test_matrix": Xt_te.shape,
                      "missing": bool(np.isnan(Xt_te).any())}
pd.DataFrame(checks).T

## Step 8 – Viva summary

In [ ]:
print(f'''
VALIDATION   5-fold StratifiedGroupKFold by Order Id | orders shared between train/val folds: {int(folds["orders_in_both"].sum())}
FOLD BALANCE val mean days {folds["val_mean_days"].min():.3f}-{folds["val_mean_days"].max():.3f} (whole train {y.mean():.3f})
GROUPING     KNN plain KFold MAE {plain_s["MAE"]:.4f} vs grouped {group_s["MAE"]:.4f} (plain is optimistic)
SKEWED       {", ".join(skew_table.index[skew_table["log_applied"]]) or "none"}
LINEAR       {CHOICE["linear"]["outlier"]} + {CHOICE["linear"]["scaler"]} (CV MAE {CHOICE["linear"]["cv_MAE"]})
KNN          {CHOICE["distance (KNN)"]["outlier"]} + {CHOICE["distance (KNN)"]["scaler"]} (CV MAE {CHOICE["distance (KNN)"]["cv_MAE"]})
TREES        no scaling (HistGB MAE {tree["none"]:.4f} vs {tree["standard"]:.4f} scaled)
BASELINES    mean {baseline_table.loc["Dummy (mean)", "MAE"]:.4f} | Shipping Mode {baseline_table.loc["Shipping Mode only", "MAE"]:.4f} MAE
''')